# 运动想象 EEG 复现 · 交互式 Notebook

把 Day 1 与 Day 2 的核心步骤串成可逐步运行的单元，每个数字和图都从你自己的环境里实时算出来。

运行前请确认：
- 已激活 `.venv`（`D:\WorkBuddy_Projects\eeg-reproduce\.venv`）
- 在 VS Code 里选择该 interpreter 作为 notebook 内核
- `data/eegmmidb/.../S001R03.edf` 与 `processed/sub001_epochs-epo.fif` 已存在
  （分别跑过 `python src/plot_raw.py` 与 `python src/preprocess.py --runs 3 4` 即生成）


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import mne

print("mne", mne.__version__, "| numpy", np.__version__)

## Day 1 · 加载 eegmmidb 并看原始信号

eegmmidb 没有左右腿，只有左拳 / 右拳（run 3/7/11 真实、4/8/12 想象）。
我们先读一条 run 3（左拳）数据，画前 5 秒 FC 区 8 通道波形。


In [ ]:
from mne.datasets import eegbci

data_dir = "data"
f = eegbci.load_data(1, 3, path=data_dir, update_path=True)
raw = mne.io.read_raw_edf(f[0], preload=True)
eegbci.standardize(raw)
try:
    montage = mne.channels.make_standard_montage("colin27_1005")
except Exception:
    montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage)

chans = ['FC5','FC3','FC1','FCz','FC2','FC4','FC6','C5']
data, times = raw.copy().pick(chans).get_data(return_times=True)

fig, axes = plt.subplots(len(chans), 1, figsize=(11, 7), sharex=True)
for ax, ch, d in zip(axes, chans, data):
    ax.plot(times, d * 1e6, lw=0.6)
    ax.set_ylabel(ch, rotation=0, labelpad=28, fontsize=9, va='center')
axes[-1].set_xlabel("time (s)")
fig.suptitle("Day1 原始 EEG 波形（FC/C 区 8 通道，单位 μV）")
plt.tight_layout()
plt.show()

In [ ]:
psd = raw.compute_psd(fmin=1, fmax=45, n_fft=1024)
psd.plot()
plt.show()

## LSL 接收示例（脑机实验的实时同步协议）

完整链路是 `OpenBCI GUI -> LSL 推流 -> Python 订阅`。核心三步：

```python
from pylsl import StreamInlet, resolve_streams
streams = resolve_streams()          # 1. 发现设备推的流
inlet = StreamInlet(streams[0])      # 2. 建立入站
samples, ts = inlet.pull_chunk()      # 3. 拉取一段样本
```

本地自测：`python src/lsl_receive_demo.py --mode selftest`（先发一条 MockEEG 流、再接收，已验证可收 1037 个样本）。
真实设备把 `resolve_streams()` 换成按名称筛选即可。


## Day 2 · 加载已预处理好的 epoch 并看平均诱发

下面直接读取 `python src/preprocess.py --runs 3 4` 算好并保存的 `processed/sub001_epochs-epo.fif`，
省去重新做 ICA 的几分钟。想从头跑就直接运行那个脚本。


In [ ]:
fif = "processed/sub001_epochs-epo.fif"
if not os.path.exists(fif):
    print("未找到", fif, "-> 请先运行: python src/preprocess.py --runs 3 4")
else:
    epochs = mne.read_epochs(fif)
    print("epoch 数:", len(epochs))
    print("各类别数量:", {k: int((epochs.events[:,2] == v).sum()) for k, v in epochs.event_id.items()})
    evoked = epochs.average()
    fig = evoked.plot(show=False)
    plt.show()

## 小结 & 下一步

- Day 1 验证了「读数据 → 画图」的完整链路；eegmmidb 是左右**手**，不是左右腿。
- Day 2 链路：带通 1-40 Hz → ICA 去眼电 → 按 T1/T2 切 epoch → 平均诱发可见 0.3-0.5 s 的诱发峰。
- 下一步（Day 3）：在 epoch 上提取 **CSP / FBCSP** 特征，用 **LDA / SVM** 做左右拳分类并出混淆矩阵；
  真正复现「左右腿」需接入 **Zuo2025**（见 `src/download_data.py::download_zuo2025`）。
